# MLP amplification check — are the churn deltas a linear-model artifact?

The consent-churn study measured modest utility deltas on a class-weighted
**logistic-regression** client and flagged one honest caveat: a saturated linear
model might be *dampening* the churn signal. This notebook settles that.

**Design.** `mlp_amplification_study.py` reruns the entire churn grid — rate
sweep (3 regimes x 4 rates), whole-silo exits, and the count-matched H2 control —
with **two clients on identical partitions and schedules** (5 seeds, K=3, FedAvg
class-weighted, 40 rounds):

| client | config | centralized AUROC |
|---|---|---|
| logreg (original) | lr 0.1 | 0.6662 |
| **MLP** | 1 ReLU hidden layer, **64 units**, lr 0.05, He init | **0.6700** |

The MLP config was picked by a small sweep (h in {32,64} x lr in {0.05,0.1});
h=64/lr=0.05 beats the logreg ceiling by +0.004 and federates to 0.6701 at
K=3/alpha=0.5, so it is a genuinely higher-capacity averageable client, not a
handicapped one. Aggregation stays model-agnostic: the MLP is encoded as one
flat parameter vector (`federated_methods.init_theta`), so FedAvg weighting,
churn schedules, and the whole harness are unchanged.

**Validation.** The logreg arm of this rerun reproduces the published churn
table exactly (transient@70% -0.003, permanent -0.007, biased -0.021,
whole-silo pos-heavy alpha=0.1 -0.059, count-matched -0.039), so the two arms
differ only in the client model.


In [1]:
import json
from pathlib import Path
import pandas as pd

res = json.loads(Path("mlp_amplification_results.json").read_text())
s = res["summary"]
cfg = res["config"]
print(f"{len(res['runs'])} runs | seeds={cfg['seeds']} rounds={cfg['rounds']} "
      f"K={cfg['k']} models={list(cfg['models'])}")
print("no-churn baseline AUROC (mean over seeds):")
for m in ("logreg", "mlp"):
    print(f"  {m:<7s} " + "  ".join(f"alpha={a}: {v:.4f}"
          for a, v in s[m]["baseline_auroc"].items()))

200 runs | seeds=[0, 1, 2, 3, 4] rounds=40 K=3 models=['logreg', 'mlp']
no-churn baseline AUROC (mean over seeds):
  logreg  alpha=0.5: 0.6587  alpha=0.1: 0.6404
  mlp     alpha=0.5: 0.6575  alpha=0.1: 0.6290


## Experiment 1 — per-patient churn rate sweep (alpha=0.5)

Delta AUROC vs the same-seed no-churn baseline, mean (sd) over 5 seeds.
`mlp/logreg` is the amplification ratio (>1 = MLP hurt more).

In [2]:
def fmt(pair):
    m, sd = pair
    return f"{m:+.4f} ({sd:.4f})"

rows = []
for regime in ("transient", "permanent", "biased"):
    for rate in cfg["rates"]:
        a = s["logreg"][regime][str(rate)]; b = s["mlp"][regime][str(rate)]
        rows.append(dict(regime=regime, rate=rate, logreg=fmt(a), mlp=fmt(b),
                         ratio=round(b[0] / a[0], 2) if abs(a[0]) > 1e-6 else None))
pd.DataFrame(rows)

,regime,rate,logreg,mlp,ratio
0,transient,0.1,-0.0009 (0.0011),-0.0002 (0.0017),0.24
1,transient,0.3,-0.0011 (0.0016),-0.0004 (0.0019),0.37
2,transient,0.5,-0.0016 (0.0009),-0.0007 (0.0033),0.44
3,transient,0.7,-0.0029 (0.0010),-0.0037 (0.0031),1.30
4,permanent,0.1,-0.0016 (0.0022),-0.0026 (0.0030),1.62
5,permanent,0.3,-0.0024 (0.0025),-0.0035 (0.0025),1.45
6,permanent,0.5,-0.0048 (0.0030),-0.0054 (0.0032),1.12
7,permanent,0.7,-0.0070 (0.0045),-0.0082 (0.0049),1.17
8,biased,0.1,-0.0090 (0.0058),-0.0076 (0.0048),0.85
9,biased,0.3,-0.0189 (0.0063),-0.0175 (0.0070),0.92


## Experiments 2 & 3 — whole-silo departure and the H2 isolation test

In [3]:
rows = []
for a in cfg["alphas"]:
    for key, label in ((f"whole_silo_heavy_a{a}", "whole-silo exit, pos-heavy"),
                       (f"whole_silo_light_a{a}", "whole-silo exit, pos-light"),
                       (f"matched_a{a}", "count-matched random (control)")):
        rows.append(dict(alpha=a, condition=label,
                         logreg=fmt(s["logreg"][key]), mlp=fmt(s["mlp"][key])))
pd.DataFrame(rows)

,alpha,condition,logreg,mlp
0,0.5,"whole-silo exit, pos-heavy",-0.0097 (0.0156),-0.0110 (0.0173)
1,0.5,"whole-silo exit, pos-light",+0.0003 (0.0010),+0.0007 (0.0024)
2,0.5,count-matched random (control),-0.0089 (0.0103),-0.0095 (0.0093)
3,0.1,"whole-silo exit, pos-heavy",-0.0593 (0.0213),-0.0637 (0.0251)
4,0.1,"whole-silo exit, pos-light",-0.0104 (0.0206),-0.0035 (0.0066)
5,0.1,count-matched random (control),-0.0391 (0.0463),-0.0506 (0.0607)


## Findings

**1. No amplification: the churn deltas are NOT a linear-model artifact.**
Across all twelve rate-sweep cells the MLP deltas match logreg within one
standard deviation (ratios 0.2-1.6, and the *biased* regime — the one that
matters — is ~0.9x, i.e. marginally *smaller* on the MLP). Silo-level effects
are likewise unchanged: pos-heavy exit at alpha=0.1 costs -0.064 on the MLP vs
-0.059 on logreg. The caveat raised in the churn study is resolved in the
direction that *strengthens* the result: the cost structure of consent churn is
**model-independent** at this capacity.

**2. H2 survives the model swap.** On the MLP, the pos-heavy whole-silo exit
(-0.064) still dominates the count-matched random control (-0.051) at identical
headcount, and dwarfs the pos-light exit (-0.004) — a ~18x who-leaves gap.
"Who leaves > how many leave" is not a property of logistic regression.

**3. Why nothing amplifies: the task ceiling, not the model class, bounds the
deltas.** The MLP lifts the averageable ceiling only 0.666 -> 0.670, against a
tree ceiling of 0.677 and a published SOTA band of 0.667-0.70. There is simply
little headroom on this task for *any* model to lose dramatically more under
distribution shift. The honest statement for the dissertation: on a real
clinical task at its signal ceiling, per-patient consent churn stays cheap and
distribution-shifting churn stays ~3-6x more expensive, regardless of client
capacity.

**4. One new observation.** The MLP's *no-churn* baseline is more fragile under
severe skew (alpha=0.1: 0.629 vs logreg's 0.640) — higher capacity overfits
skewed silos slightly faster, worth one sentence when reporting.

**Consequence for the roadmap:** the utility axis is now closed and
model-robust. Next build is the systems-cost axis (RQ1): on-chain consent
instrumentation on the per-round hook, with these same churn schedules as the
transaction-traffic generator.